# Build_Gold_Sales_Tables

Builds the Gold-layer tables for sales reporting from `silver_orders` and the shortcut reference tables (`customers`, `products`). Modeled dimensionally (one fact, three dimensions); the actual star-schema relationships are defined later, in the semantic model.

**Produces:** `Dim_Customer`, `Dim_Product` (from shortcut reference data), `Dim_Date` (generated to span the order-date range), `Fact_Sales` (one row per order, same grain as `silver_orders`).

In [ ]:
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

dim_customer_table = "gold.Dim_Customer"
dim_product_table = "gold.Dim_Product"
dim_date_table = "gold.Dim_Date"
fact_sales_table = "gold.Fact_Sales"

In [ ]:
# Dim_Customer & Dim_Product: straight passthrough from the shortcut
# reference tables — Corporate already provides curated master data, just
# select/rename to the Gold naming convention.
df_dim_customer = spark.read.table("customers").select(
    "CustomerId", "CustomerName", "Segment", "City", "Country", "Email"
)
df_dim_customer.write.format("delta").mode("overwrite").saveAsTable(dim_customer_table)

df_dim_product = spark.read.table("products").select(
    "ProductId", "ProductName", "Category", "RequiresColdChain", "UnitPrice"
)
df_dim_product.write.format("delta").mode("overwrite").saveAsTable(dim_product_table)

print(f"-> {dim_customer_table}: {df_dim_customer.count()} rows")
print(f"-> {dim_product_table}: {df_dim_product.count()} rows")

In [ ]:
# Dim_Date: generated to span the full range of order dates (plus a small
# buffer), not hardcoded, so it stays correct as new order dates arrive.
date_bounds = spark.sql(
    "SELECT MIN(OrderDate) AS min_date, MAX(OrderDate) AS max_date FROM silver_orders"
).collect()[0]

df_dim_date = (
    spark.sql(f"""
        SELECT explode(sequence(
            date_sub(to_date('{date_bounds.min_date}'), 7),
            date_add(to_date('{date_bounds.max_date}'), 7),
            interval 1 day
        )) AS Date
    """)
    .withColumn("Year", F.year("Date"))
    .withColumn("Quarter", F.quarter("Date"))
    .withColumn("Month", F.month("Date"))
    .withColumn("MonthName", F.date_format("Date", "MMMM"))
    .withColumn("Weekday", F.date_format("Date", "EEEE"))
    .withColumn("IsWeekend", F.dayofweek("Date").isin(1, 7))
)

df_dim_date.write.format("delta").mode("overwrite").saveAsTable(dim_date_table)
print(f"-> {dim_date_table}: {df_dim_date.count()} rows")

In [ ]:
# Fact_Sales: one row per order, same grain as silver_orders. Only the
# reporting-relevant columns are kept; OrderDate is cast to a plain date to
# match Dim_Date's join key.
df_fact_sales = spark.read.table("silver_orders").select(
    "OrderId",
    "CustomerId",
    "ProductId",
    F.to_date("OrderDate").alias("OrderDate"),
    "Quantity",
    "UnitPrice",
    "TotalAmount",
    "Status",
    "DeliveryRegion",
)

df_fact_sales.write.format("delta").mode("overwrite").saveAsTable(fact_sales_table)
print(f"-> {fact_sales_table}: {df_fact_sales.count()} rows")